<a href="https://colab.research.google.com/github/keerthisingh-analyst/chatgpt_review_analysis/blob/main/ChatGpt_Review_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from textblob import TextBlob
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings('ignore')
#

In [ ]:
Link = "/content/chatgpt_reviews.xlsx"
df = pd.read_excel(Link)
df.head()

In [ ]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['review'] = df['review'].fillna(" ")
df.isnull().sum()

**Sentiment Analysis**

In [ ]:
def get_sentiment_polarity(text):
    if pd.isna(text) or text == '':
        return 0

    blob = TextBlob(str(text))
    return blob.sentiment.polarity

df['sentiment_polarity'] = df['review'].apply(get_sentiment_polarity)

df.head(10)

In [ ]:
def get_sentiment_category(polarity):
    if polarity > 0.1:
        return 'Positive'
    elif polarity < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_category'] = df['sentiment_polarity'].apply(get_sentiment_category)

df.head()

In [ ]:
def get_sentiment_subjectivity(text):
    if pd.isna(text) or text == '':
        return 0

    blob = TextBlob(str(text))
    return blob.sentiment.subjectivity

df['sentiment_subjectivity'] = df['review'].apply(get_sentiment_subjectivity)

df.head()

In [ ]:
df['sentiment_category'].value_counts()

In [ ]:
sentiment_category = df['sentiment_category'].value_counts()

sns.barplot(
    x=sentiment_category.index,
    y=sentiment_category.values
)

plt.xlabel('Sentiment Category')
plt.ylabel('Count')
plt.title('Sentiment Distribution on Review Analysis')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(
    df['sentiment_subjectivity'],
    bins=20,
    alpha=0.7,
    color='lightcoral',
    edgecolor='black'
)

plt.axvline(
    df['sentiment_subjectivity'].mean(),
    color='blue',
    linestyle='--',
    label=f"Mean: {df['sentiment_subjectivity'].mean():.3f}"
)

plt.title(
    "Distribution of Sentiment Subjectivity Scores",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Subjectivity Score (0 to 1)")
plt.ylabel("Frequency")
plt.legend()

plt.show()

In [ ]:
# # 3. Rating Distribution by Sentiment Category
sentiment_rating = df.groupby(['sentiment_category', 'ratings']).size().unstack(fill_value=0)
sentiment_rating.plot(kind='bar', stacked=True)
plt.title('Rating Distribution by Sentiment Category', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment Category')
plt.ylabel('Number of Reviews')
plt.legend(title='Rating', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.show()

**Analyzing Positive Features**

In [ ]:
positive_reviews = df[df['sentiment_category'] == 'Positive']['review']

In [ ]:
positive_reviews

Extract Positive Keywords and Phrases

In [ ]:
import re
# Extract positive keywords and phrases

all_text = ' '.join(positive_reviews.astype(str)).lower()

# Clean text
cleaned_text = re.sub(r'[^a-zA-Z\s]', '', all_text)
words = cleaned_text.split()

# Remove all words whose length is < 3
words = [word for word in words if len(word) > 2]

# Get phrases (bigrams and trigrams)
phrases = []
for i in range(len(words) - 1):
    if len(words[i]) > 2 and len(words[i+1]) > 2:  # Skip short words
        phrases.append(f"{words[i]} {words[i+1]}")

for i in range(len(words) - 2):
    if all(len(word) > 2 for word in words[i:i+3]):  # Skip short words
        phrases.append(f"{words[i]} {words[i+1]} {words[i+2]}")

phrase_freq = Counter(phrases)

pos_phrases = phrase_freq.most_common(10)

In [ ]:
pos_phrases

In [ ]:
from ipywidgets.widgets.widget_int import Color
sns.barplot(x=[phrase[0] for phrase in pos_phrases], y=[phrase[1] for phrase in pos_phrases], color='green')
plt.xlabel('Positive Phrases')
plt.ylabel('Frequency')
plt.title('Most Common Positive Phrases')
plt.xticks(rotation=45)
plt.show()


In [ ]:
Negative_reviews = df[df['sentiment_category'] == "Negative"]['review']

In [ ]:
Negative_reviews

In [ ]:
import re
# Extract positive keywords and phrases

all_text = ' '.join(Negative_reviews.astype(str)).lower()

# Clean text
cleaned_text = re.sub(r'[^a-zA-Z\s]', '', all_text)
words = cleaned_text.split()

# Remove all words whose length is < 3
words = [word for word in words if len(word) > 2]

# Get phrases (bigrams and trigrams)
phrases = []
for i in range(len(words) - 1):
    if len(words[i]) > 2 and len(words[i+1]) > 2:  # Skip short words
        phrases.append(f"{words[i]} {words[i+1]}")

for i in range(len(words) - 2):
    if all(len(word) > 2 for word in words[i:i+3]):  # Skip short words
        phrases.append(f"{words[i]} {words[i+1]} {words[i+2]}")

phrase_freq = Counter(phrases)

neg_phrases = phrase_freq.most_common(10)

In [ ]:
neg_phrases

In [ ]:
sns.barplot(x=[phrase[0] for phrase in neg_phrases], y=[phrase[1] for phrase in neg_phrases], color='red')
plt.xlabel('Negative Phrases')
plt.ylabel('Frequency')
plt.title('Most Common Negative Phrases')
plt.xticks(rotation=45)
plt.show()

In [ ]:
neutral_reviews = df[df['sentiment_category'] == "Neutral"]['review']

In [ ]:
neutral_reviews

In [ ]:
all_text = ' '.join(neutral_reviews.astype(str)).lower()

# Clean text
cleaned_text = re.sub(r'[^a-zA-Z\s]', '', all_text)
words = cleaned_text.split()

# Remove all words whose length is < 3
words = [word for word in words if len(word) > 2]

# Get phrases (bigrams and trigrams)
phrases = []
for i in range(len(words) - 1):
    if len(words[i]) > 2 and len(words[i+1]) > 2:  # Skip short words
        phrases.append(f"{words[i]} {words[i+1]}")

for i in range(len(words) - 2):
    if all(len(word) > 2 for word in words[i:i+3]):  # Skip short words
        phrases.append(f"{words[i]} {words[i+1]} {words[i+2]}")

phrase_freq = Counter(phrases)

neu_phrases = phrase_freq.most_common(10)

In [ ]:
neu_phrases

In [ ]:
sns.barplot(x=[phrase[0] for phrase in neu_phrases], y=[phrase[1] for phrase in neu_phrases], color='pink')
plt.xlabel('Neutral Phrases')
plt.ylabel('Frequency')
plt.title('Most Common Neutral Phrases')
plt.xticks(rotation=45)
plt.show()

In [ ]:
issue_categories = {
    'Performance': ['slow', 'lag', 'crash', 'freeze', 'glitch', 'bug', 'error'],
    'Features': ['missing', 'lack', 'need', 'want', 'should', 'feature'],
    'User Experience': ['confusing', 'difficult', 'hard', 'complicated', 'interface'],
    'Accuracy': ['wrong', 'incorrect', 'bad', 'poor', 'inaccurate', 'mistake'],
    'Subscription': ['expensive', 'price', 'cost', 'pay', 'subscription', 'money']
}

category_counts = {}
for category, keywords in issue_categories.items():
    count = 0
    for review in Negative_reviews:
        if any(keyword in str(review).lower() for keyword in keywords):
            count += 1
    category_counts[category] = count

categories = list(category_counts.keys())
counts = list(category_counts.values())

plt.bar(categories, counts, color='salmon', alpha=0.8)
plt.title('Categorization of Negative Issues', fontsize=14, fontweight='bold')
plt.xlabel('Issue Category')
plt.ylabel('Number of Reviews Mentioning')
plt.xticks(rotation=45)

plt.show()

In [ ]:
neutral_texts = neutral_reviews

keyword_map = {
    'Feature Request': ['add', 'include', 'feature', 'option', 'setting', 'would like', 'wish', 'hope'],
    'Question': ['how', 'what', 'why', 'when', 'where', 'can', 'does', 'is', '?'],
    'Positive-leaning': ['good', 'nice', 'ok', 'fine', 'decent', 'works'],
    'Negative-leaning': ['but', 'however', 'issue', 'problem', 'concern', 'could be better']
}

neutral_categories = []

for text in neutral_texts:
    text_lower = str(text).lower()
    categories = []

    for label, keywords in keyword_map.items():
        if any(keyword in text_lower for keyword in keywords):
            categories.append(label)

    if not categories:
        categories.append('Truly Neutral')

    neutral_categories.extend(categories)

# Count categories
category_counts = Counter(neutral_categories)

if category_counts:
    plt.figure()

    categories = list(category_counts.keys())
    counts = list(category_counts.values())

    plt.bar(categories, counts, color='orange', alpha=0.8)
    plt.title('Categorization of Neutral Reviews', fontsize=14, fontweight='bold')
    plt.xlabel('Category')
    plt.ylabel('Number of Mentions')
    plt.xticks(rotation=45)

    plt.show()

**Review Evolution Over Time**

In [ ]:
df['review_month'] = df['review_date'].dt.to_period('M')
df['review_quater'] = df['review_date'].dt.to_period('Q')


In [ ]:
monthly_counts = df.groupby('review_month').size()
plt.plot(monthly_counts.index.astype(str), monthly_counts.values,
         marker='o', linewidth=2, markersize=8, color='blue')

plt.title('Review Volume Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
df['review_date'] = pd.to_datetime(df['review_date'])

sentiment_over_time = df.groupby([df['review_date'].dt.to_period('M'), 'sentiment_category'])\
                        .size().unstack(fill_value=0)

sentiment_over_time.index = sentiment_over_time.index.to_timestamp()
sentiment_over_time.plot()

plt.title('Sentiment Trends Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Number of Reviews')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.xticks(rotation=45)
plt.legend(title='Sentiment')

plt.show()

In [ ]:
df['review_date'] = pd.to_datetime(df['review_date'])

sentiment_over_time = df.groupby([df['review_date'].dt.to_period('M'), 'ratings'])\
                        .size().unstack(fill_value=0)

sentiment_over_time.index = sentiment_over_time.index.to_timestamp()
sentiment_over_time.plot()

plt.title('Rating Trends Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Number of Reviews')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.xticks(rotation=45)
plt.legend(title='Sentiment')

plt.show()

In [ ]:
import os

# Ensure the current working directory is /content/
os.chdir('/content/')

# Create a .gitignore file to exclude unnecessary files
!echo -e "/.config/\n/sample_data/\n*.xlsx\n*.ipynb_checkpoints/\n.git/" > .gitignore

# Initialize Git repository (harmless if already done)
!git init

# Configure Git user (replace with your details)
# IMPORTANT: Replace "your_email@example.com" and "Your Name" with your GitHub credentials
!git config user.email "your_email@example.com"
!git config user.name "Your Name"

# Add the current notebook and .gitignore to the repository
# IMPORTANT: Please manually save your current notebook as 'chatgpt_review_analysis.ipynb'
# in the /content/ directory BEFORE running this cell.
!git add chatgpt_review_analysis.ipynb .gitignore

# Commit the changes (|| true prevents error if nothing new to commit)
!git commit -m "Initial commit of Colab notebook with .gitignore" || true

# Remove existing remote 'origin' if it exists to avoid 'remote origin already exists' error
!git remote remove origin || true

# Add the GitHub remote origin with your Personal Access Token (PAT) for authentication
# IMPORTANT: Replace <YOUR_GITHUB_TOKEN> with your actual GitHub Personal Access Token.
# To generate a PAT, go to GitHub settings > Developer settings > Personal access tokens.
!git remote add origin https://<YOUR_GITHUB_TOKEN>@github.com/keerthisingh-analyst/chatgpt_review_analysis.git

# Set the branch name to 'main'
!git branch -M main

# Push the changes
!git push -u origin main